### Explanation before notebook title and runbook overview

This opening markdown cell states the purpose and execution order of the submission notebook. It is placed first so a reviewer immediately understands that the notebook is a reproducible runbook rather than only a collection of ad-hoc experiments.

# Assignment 2: Bus-Level Load Forecasting Submission Notebook

This notebook is ordered as a submission runbook: validate inputs, train/validate Method 1, train both Method 2 zone models, run one shared bus-share allocation pass, then compare metrics and list deliverables.

### Explanation before README/submission checklist

This markdown cell summarizes the assignment requirements that the rest of the notebook must satisfy: forecast schema, row coverage, metrics, baselines, leakage control, and deliverables. It is included early so every later validation step can be tied back to a concrete requirement.

## 1. README Checklist / Submission Checklist

Required final checks:

- Forecast schema: `model_name`, `forecast_created_at`, `target_date`, `he`, `bus_id`, `zone_id`, `predict_pd`.
- Row count: each forecast task must cover every provided 2025 bus-hour target row.
- Metrics: include model and baseline rows for bus-level and zone-level evaluation.
- Report: include modeling choices, leakage controls, validation, limitations, and final model selection.
- AI log: include how AI assistance was used.

### Explanation before setup section header

This section header separates environment setup from modeling logic. The separation makes it easier for a reviewer to confirm that imports, paths, and module reloads are handled before any data or model steps run.

## 2. Setup and Imports

### Explanation before setup/imports code

This code cell imports the notebook dependencies and dynamically locates the Python modules, input data, and output/cache folders. Keeping paths relative makes the submission easier to clone or move to another machine. It also reloads the project modules so rerunning the notebook uses the latest local code instead of a stale Python import cache.

In [ ]:
import sys
from pathlib import Path
import importlib

import pandas as pd
import pyarrow.parquet as pq

# Keep module, input, and output/cache paths relative so the submission can be moved or cloned.
module_candidates = [
    Path("../outputs"),
    Path("../code"),
    Path("."),
    Path("outputs"),
    Path("code"),
    Path("assignment2_load_forecasting/outputs"),
    Path("assignment2_load_forecasting/code"),
    Path("ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("ecesis-assignment-submission/assignment2_load_forecasting/code"),
    Path("../ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("../ecesis-assignment-submission/assignment2_load_forecasting/code"),
    Path("../../ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("../../ecesis-assignment-submission/assignment2_load_forecasting/code"),
    Path("Summer2026-main/ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("Summer2026-main/ecesis-assignment-submission/assignment2_load_forecasting/code"),
]
for candidate in module_candidates:
    if (candidate / "assignment2_method2_zone_share_transformer.py").exists():
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not locate Assignment 2 Python module directory from relative paths.")

artifact_candidates = [
    Path("../outputs"),
    MODULE_DIR,
    Path("outputs"),
    Path("assignment2_load_forecasting/outputs"),
    Path("ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("../ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("../../ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
    Path("Summer2026-main/ecesis-assignment-submission/assignment2_load_forecasting/outputs"),
]
for candidate in artifact_candidates:
    if (candidate / "assignment2_metrics.csv").exists() or (candidate / "assignment2_next_day_forecast_2025.parquet").exists():
        NOTEBOOK_DIR = candidate
        break
else:
    NOTEBOOK_DIR = MODULE_DIR

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import assignment2_method1_direct_bus_transformer as method1_module
import assignment2_method2_zone_share_transformer as method2_module
import assignment2_weather as weather_module

method1_module = importlib.reload(method1_module)
method2_module = importlib.reload(method2_module)
weather_module = importlib.reload(weather_module)

run_direct_bus_transformer = method1_module.run_direct_bus_transformer
run_direct_bus_data_stage = method1_module.run_direct_bus_data_stage
run_direct_bus_training_stage = method1_module.run_direct_bus_training_stage
run_direct_bus_forecast_stage = method1_module.run_direct_bus_forecast_stage
run_direct_bus_validation_stage = method1_module.run_direct_bus_validation_stage

run_zone_share_transformer = method2_module.run_zone_share_transformer
run_method2_zone_forecasting_stage = method2_module.run_method2_zone_forecasting_stage
run_method2_bus_share_stage = method2_module.run_method2_bus_share_stage
run_method2_validation_stage = method2_module.run_method2_validation_stage
run_combined_method2_bus_share_stage = method2_module.run_combined_method2_bus_share_stage

download_zone_weather = weather_module.download_zone_weather
weather_paths = weather_module.weather_paths
weather_model_paths = weather_module.weather_model_paths
run_weather_zone_training_stage = weather_module.run_weather_zone_training_stage
run_weather_bus_share_stage = weather_module.run_weather_bus_share_stage
run_weather_validation_stage = weather_module.run_weather_validation_stage

print(f"Module directory: {MODULE_DIR}")
print(f"Artifact/cache directory: {NOTEBOOK_DIR}")



### Explanation before data/path validation section header

This header introduces the input and output validation step. It is important to check file paths and expected schemas before starting expensive forecast generation or model validation.

## 3. Data and Output Path Validation

### Explanation before data and output validation code

This cell verifies that the project root, Assignment 2 data directory, required parquet inputs, expected 2025 bus-row count, and required forecast schema are available. This check is important because the full forecasts are very large; catching a missing input file or wrong path early prevents a long run from failing later.

In [ ]:
project_root = method2_module.find_project_root()
assignment2_data_dir = method2_module.find_assignment2_data_dir(project_root)
expected_2025_bus_rows = method2_module.expected_2025_bus_rows(assignment2_data_dir)
forecast_schema = method2_module.FORECAST_COLUMNS

required_data_files = []
for year in [2022, 2023, 2024, 2025]:
    required_data_files.append((f"bus_load_{year}", method2_module.bus_path(assignment2_data_dir, year)))
    required_data_files.append((f"zone_load_{year}", method2_module.zone_path(assignment2_data_dir, year)))

data_file_checks = pd.DataFrame(
    [
        {
            "item": name,
            "path": str(path),
            "exists": path.exists(),
            "rows": pq.ParquetFile(path).metadata.num_rows if path.exists() else None,
        }
        for name, path in required_data_files
    ]
)

submission_checklist = pd.DataFrame(
    [
        {"check": "forecast_schema", "expected": forecast_schema},
        {"check": "forecast_row_count_per_task", "expected": expected_2025_bus_rows},
        {"check": "metrics", "expected": "CSV with task, level, model_name, n, mae, rmse, wmape, sum_actual"},
        {"check": "report", "expected": "summary markdown report"},
        {"check": "ai_log", "expected": "AI usage log markdown"},
    ]
)

print(f"Project root: {project_root}")
print(f"Assignment 2 data dir: {assignment2_data_dir}")
print(f"Expected 2025 bus forecast rows per task: {expected_2025_bus_rows:,}")
display(submission_checklist)
data_file_checks

### Explanation before baseline section description

This markdown cell introduces the baseline logic used throughout the notebook. It explains the strict forecast cutoff interpretation so the reviewer can see why the pipeline avoids D-1 actual load for next-day forecasts created at D-1 00:01.

## 4. Baselines

The notebook tracks the README baselines throughout Method 1 and Method 2. The strict cutoff interpretation is used: for a next-day forecast created at `D-1 00:01`, D-1 actual load is not used as a feature.

### Explanation before baseline summary code

This cell documents the baseline models used for comparison. The baselines provide simple, auditable references for the Transformer results: previous-week load for next-day, previous-year load for next-month, and a historical bus/hour/weekday average. These baselines also help show whether the neural model adds value beyond simple seasonal copying.

In [ ]:
baseline_summary = pd.DataFrame(
    [
        {
            "baseline": "previous-week",
            "used_for": "next_day",
            "level": "bus and zone metrics",
            "definition": "same bus/hour from D-7; no D-1 actual is used under the strict D-1 00:01 cutoff",
        },
        {
            "baseline": "previous-year",
            "used_for": "next_month",
            "level": "bus and zone metrics",
            "definition": "same bus/calendar date/hour from the previous year",
        },
        {
            "baseline": "bus-hour-weekday historical average",
            "used_for": "next_day and next_month",
            "level": "bus and zone metrics",
            "definition": "2022-2024 average by bus, hour-ending, and weekday",
        },
    ]
)
baseline_summary

### Explanation before Method 1 overview

This markdown cell describes the direct bus-level Transformer approach. It explains that Method 1 works directly at the bus level with sampled historical rows, zone embeddings, hashed bus embeddings, and residual learning relative to historical baselines.

## 5. Method 1: Direct Bus Transformer


Method 1 is a direct bus-level forecasting model. Instead of first forecasting zone load and then allocating it to buses, this method tries to predict each bus-hour load directly. However, the raw bus-level table is extremely sparse and noisy: many buses are zero-load buses, newly appearing buses, retired buses, or buses with missing pd values. If the model is trained directly on these raw sparse rows, it can learn a trivial zero-heavy pattern and give too much importance to inactive or unreliable buses. That would make the direct bus model unstable and less useful for real load forecasting.

For this reason, I first prepare the bus-level data before training. The preprocessing step audits missingness, separates partial-missing buses from all-missing buses, builds historical baselines, constructs historical bus profiles, adds missing-data indicators, and samples training rows from every available 2022-2024 training date. After preprocessing, the model is trained on a cleaner and more informative bus-level dataset. The Transformer uses calendar features, historical baseline features, zone embeddings, and stable hashed bus-id embeddings. It predicts a log residual relative to the historical baseline rather than predicting raw load directly. This makes the model focus on correcting the baseline instead of relearning the entire load level from sparse raw data.

The section is split into data preparation, model training, forecast writing, and validation. The validation table must be all True before this method is treated as submittable.


### Explanation before Method 1 configuration header

This header marks the cell where Method 1 run flags and hyperparameters are defined. Keeping configuration separate from execution makes it clear how to reproduce or intentionally rebuild Method 1.

### 5.A Method 1 Configuration

### Explanation before Method 1 configuration code

This cell collects Method 1 settings in one place. The run flags are set to reuse existing caches and outputs by default, which is safer for a formal submission because it avoids accidentally retraining or overwriting validated files. The hyperparameters describe the direct bus-level Transformer setup if a rebuild is intentionally enabled.

In [ ]:
USE_MISSING_INDICATORS = True  # Whether to include baseline_missing/profile_missing as Method 1 input features.
RUN_METHOD1_DATA_PREP = False  # Formal submission default: reuse the existing Method 1 data cache.
RUN_METHOD1_TRAINING = False  # Formal submission default: reuse existing Method 1 checkpoints.
RUN_METHOD1_FORECAST = False  # Formal submission default: reuse existing Method 1 forecast files.

METHOD1_TRAINING_YEARS = [2022, 2023, 2024]  # Use all available pre-2025 years for Method 1 training samples.
METHOD1_ROWS_PER_DATE = 20000  # 10GB GPU / 64GB RAM setting; increase carefully if data prep and training remain stable.
METHOD1_BUS_HASH_BUCKETS = 262_144  # Stable hashed bus-id embedding buckets; about 32 MB at d_model=32.

METHOD1_SHARED_MAX_EPOCHS = 50  # Maximum epochs for the shared global trunk; early stopping may stop earlier.
METHOD1_HEAD_MAX_EPOCHS = 20  # Maximum epochs for the zone-specific heads after the trunk is frozen.
METHOD1_VALIDATION_FRACTION = 0.2  # Last 20% of sampled training rows is used as chronological validation.
METHOD1_EARLY_STOPPING_PATIENCE = 8  # Stop if monitored validation loss does not improve for this many epochs.
METHOD1_EARLY_STOPPING_MIN_DELTA = 1e-4  # Minimum monitored-loss improvement required to reset early stopping.
METHOD1_SHARED_LEARNING_RATE = 1e-3  # AdamW initial learning rate for the shared trunk.
METHOD1_HEAD_LEARNING_RATE = 1e-3  # AdamW initial learning rate for zone-specific heads.
METHOD1_SHARED_WEIGHT_DECAY = 1e-4  # L2-style regularization for the shared trunk optimizer.
METHOD1_HEAD_WEIGHT_DECAY = 1e-4  # L2-style regularization for the zone-head optimizer.
METHOD1_DROPOUT = 0.1  # Transformer dropout for Method 1.
METHOD1_USE_REDUCE_LR_ON_PLATEAU = True  # Set True to reduce LR when validation loss plateaus.
METHOD1_LR_SCHEDULER_FACTOR = 0.5  # ReduceLROnPlateau multiplies LR by this factor.
METHOD1_LR_SCHEDULER_PATIENCE = 4  # Plateau epochs before reducing LR.
METHOD1_LR_SCHEDULER_MIN_LR = 1e-5  # Lower bound for scheduler-reduced LR.
METHOD1_BATCH_SIZE = 32768  # 10GB GPU setting; reduce to 16384 if CUDA OOM occurs.
METHOD1_MIN_ZONE_SAMPLES = 20000  # Minimum rows required to train a zone-specific head.
METHOD1_D_MODEL = 32  # Transformer hidden size.
METHOD1_NHEAD = 4  # Attention heads; must divide METHOD1_D_MODEL.
METHOD1_NUM_LAYERS = 2  # Number of Transformer encoder layers.
METHOD1_DIM_FEEDFORWARD = 96  # Feed-forward layer width inside each Transformer encoder layer.

import assignment2_method1_direct_bus_transformer as method1_module  # Reload Method 1 helper module.
method1_module = importlib.reload(method1_module)  # Make sure notebook uses the newest staged Method 1 logic.
run_direct_bus_data_stage = method1_module.run_direct_bus_data_stage  # Rebind data-prep stage.
run_direct_bus_training_stage = method1_module.run_direct_bus_training_stage  # Rebind training stage.
run_direct_bus_forecast_stage = method1_module.run_direct_bus_forecast_stage  # Rebind forecast-output stage.
run_direct_bus_validation_stage = method1_module.run_direct_bus_validation_stage  # Rebind validation runner.
method1_paths = method1_module.direct_transformer_paths(NOTEBOOK_DIR)  # Locate Method 1 output files.
method1_stage_paths = method1_module.direct_stage_paths(NOTEBOOK_DIR)  # Locate staged cache/checkpoint files.
method1_outputs_present = method1_module.direct_transformer_outputs_complete(method1_paths)  # Check final forecast outputs.
method1_data_present = method1_module.direct_data_stage_complete(
    method1_stage_paths,
    USE_MISSING_INDICATORS,
    METHOD1_ROWS_PER_DATE,
    METHOD1_TRAINING_YEARS,
    METHOD1_BUS_HASH_BUCKETS,
)  # Check prepared data cache and hyperparameter compatibility.
method1_checkpoints_present = method1_module.direct_training_stage_complete(method1_stage_paths, method1_paths)  # Check model checkpoints.
method1_artifacts = {'method_name': 'method1_direct_bus_transformer', 'paths': method1_paths, 'stage_paths': method1_stage_paths, 'available': method1_outputs_present, 'rebuilt': False}

print(f"Use missing indicators: {USE_MISSING_INDICATORS}")
print(f"Run data prep now: {RUN_METHOD1_DATA_PREP}")
print(f"Run training now: {RUN_METHOD1_TRAINING}")
print(f"Run forecast output now: {RUN_METHOD1_FORECAST}")
print(f"Method 1 training years: {METHOD1_TRAINING_YEARS}")
print(f"Method 1 rows per date: {METHOD1_ROWS_PER_DATE:,}")
print(f"Method 1 bus hash buckets: {METHOD1_BUS_HASH_BUCKETS:,}")
print(f"Method 1 batch size: {METHOD1_BATCH_SIZE:,}")
print(f"Method 1 target mode: {method1_module.DIRECT_TARGET_MODE}")
print(f"Method 1 loss weighting: {method1_module.DIRECT_SAMPLE_WEIGHT_MODE}")
print(f"Method 1 LR scheduler: {METHOD1_USE_REDUCE_LR_ON_PLATEAU}")
print(f"Prepared data cache present: {method1_data_present}")
print(f"Model checkpoints present: {method1_checkpoints_present}")
print(f"Final forecast outputs complete: {method1_outputs_present}")
method1_stage_paths


### Explanation before Method 1 data-preparation header

This header introduces the optional Method 1 cache-building stage. Separating this stage helps show that data preparation, model training, forecast writing, and validation are independently controlled.

### 5.B Method 1 Data Preparation

### Explanation before Method 1 data-preparation code

This cell optionally rebuilds the Method 1 data-preparation cache. The cache includes imputation artifacts, bus profiles, feature frames, and train tables. Reusing the cache is the default because preparing sampled bus-level training data is expensive, while validation only needs to confirm that the saved artifacts are present and consistent.

This cell prepares the cached training and forecasting inputs for Method 1. The goal is not to change the model logic, but to make the raw bus-level data usable for direct bus forecasting.

The main reason this step is necessary is that bus-level load is much sparser than zone-level load. Some buses have reliable nonzero historical load, some have only occasional missing values, and some are all missing or effectively always zero. Training a direct model on the raw table without preprocessing would make the model dominated by zero or missing rows. It would also make the bus-id information unreliable, because a bus may appear in one year but not have enough historical evidence in another year.

The preprocessing handles this problem in several steps. First, it builds a missingness audit for each bus and year. Buses with partial missing values and at least some nonzero historical load are treated as recoverable. Their missing historical pd values are imputed using a time-aware fallback hierarchy: same bus and same hour around nearby historical days, same bus and same hour on the same weekday, same bus and same hour average, and then a zone-hour average multiplied by historical bus share. If none of these sources are available, the value falls back to zero. All-missing buses are not used as reliable historical profile information, because there is no evidence that their missing values represent real load.

Second, the cell builds historical baseline features for the direct model. For the next-day task, the baseline is based on the previous-week source day. For the next-month task, the baseline is based on the previous-year source day. These baselines give the model a stable starting point, so the Transformer only needs to learn the residual correction.

Third, the cell builds same-bus same-hour historical profile features. These profile features summarize the typical load behavior of each bus and hour using only allowed historical years. This gives the model a compact representation of bus-level history without loading the entire historical bus table during training.

Fourth, the cell adds model features, including hour, day-of-week, month, weekend flag, holiday flag, log baseline load, log historical profile load, and missingness indicators. It also maps each zone to an integer embedding and maps each bus id into a stable hashed bus embedding bucket. The hashed bus embedding lets the model learn bus-specific patterns without creating an enormous embedding table for every possible bus id.

Finally, the cell samples a fixed number of bus rows from each training date. This keeps the training set computationally manageable while still covering every available 2022-2024 date. The target is defined as the log residual between actual load and the historical baseline, and the sample weights give more importance to nonzero and larger-load buses. This reduces the risk that the direct bus model is overwhelmed by all-zero rows.

In [ ]:
if RUN_METHOD1_DATA_PREP:
    method1_data_artifacts = run_direct_bus_data_stage(
        force_rebuild=True,  # Rebuild imputer/profile/training-frame cache.
        use_missing_indicators=USE_MISSING_INDICATORS,
        rows_per_date=METHOD1_ROWS_PER_DATE,
        training_years=METHOD1_TRAINING_YEARS,
        bus_hash_buckets=METHOD1_BUS_HASH_BUCKETS,
    )
else:
    if method1_module.direct_data_stage_complete(
        method1_stage_paths,
        USE_MISSING_INDICATORS,
        METHOD1_ROWS_PER_DATE,
        METHOD1_TRAINING_YEARS,
        METHOD1_BUS_HASH_BUCKETS,
    ):
        method1_data_artifacts = run_direct_bus_data_stage(
            force_rebuild=False,  # Load existing prepared data cache.
            use_missing_indicators=USE_MISSING_INDICATORS,
            rows_per_date=METHOD1_ROWS_PER_DATE,
            training_years=METHOD1_TRAINING_YEARS,
            bus_hash_buckets=METHOD1_BUS_HASH_BUCKETS,
        )
    else:
        method1_data_artifacts = {'available': False, 'rebuilt': False, 'stage_paths': method1_stage_paths}
        print('Method 1 data cache is missing or was built with incompatible Method 1 settings. Set RUN_METHOD1_DATA_PREP=True to build it.')

print(f"Method 1 data stage available: {method1_data_artifacts['available']}; rebuilt: {method1_data_artifacts['rebuilt']}")
if method1_data_artifacts.get('available'):
    print(f"Training years: {method1_data_artifacts.get('training_years')}")
    print(f"Rows per date: {method1_data_artifacts.get('rows_per_date'):,}")
    print(f"Bus hash buckets: {method1_data_artifacts.get('bus_hash_buckets'):,}")
    print(f"Next-day train rows: {method1_data_artifacts.get('next_day_train_rows'):,}")
    print(f"Next-month train rows: {method1_data_artifacts.get('next_month_train_rows'):,}")
method1_data_artifacts


### Explanation before Method 1 training header

This header introduces the Method 1 training stage. The notebook keeps it separate because GPU training is expensive and should only run when the user intentionally enables it.

### 5.C Method 1 Training

### Explanation before Method 1 training code

This cell optionally trains the direct bus Transformer checkpoints for next-day and next-month forecasting. It is separated from data preparation and forecast writing so each expensive stage can be controlled independently. The default is to reuse trained checkpoints, which keeps the notebook reproducible without forcing a GPU training run every time.

In [ ]:
if RUN_METHOD1_TRAINING:
    method1_training_artifacts = run_direct_bus_training_stage(
        force_rebuild=True,  # Retrain and overwrite Method 1 checkpoints/training log.
        use_missing_indicators=USE_MISSING_INDICATORS,
        shared_max_epochs=METHOD1_SHARED_MAX_EPOCHS,
        head_max_epochs=METHOD1_HEAD_MAX_EPOCHS,
        validation_fraction=METHOD1_VALIDATION_FRACTION,
        early_stopping_patience=METHOD1_EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=METHOD1_EARLY_STOPPING_MIN_DELTA,
        shared_learning_rate=METHOD1_SHARED_LEARNING_RATE,
        head_learning_rate=METHOD1_HEAD_LEARNING_RATE,
        shared_weight_decay=METHOD1_SHARED_WEIGHT_DECAY,
        head_weight_decay=METHOD1_HEAD_WEIGHT_DECAY,
        batch_size=METHOD1_BATCH_SIZE,
        min_zone_samples=METHOD1_MIN_ZONE_SAMPLES,
        d_model=METHOD1_D_MODEL,
        nhead=METHOD1_NHEAD,
        num_layers=METHOD1_NUM_LAYERS,
        dim_feedforward=METHOD1_DIM_FEEDFORWARD,
        dropout=METHOD1_DROPOUT,
        use_reduce_lr_on_plateau=METHOD1_USE_REDUCE_LR_ON_PLATEAU,
        lr_scheduler_factor=METHOD1_LR_SCHEDULER_FACTOR,
        lr_scheduler_patience=METHOD1_LR_SCHEDULER_PATIENCE,
        lr_scheduler_min_lr=METHOD1_LR_SCHEDULER_MIN_LR,
    )
else:
    if method1_module.direct_training_stage_complete(method1_stage_paths, method1_paths):
        method1_training_artifacts = run_direct_bus_training_stage(
            force_rebuild=False,  # Load existing checkpoints/training log.
            use_missing_indicators=USE_MISSING_INDICATORS,
            shared_max_epochs=METHOD1_SHARED_MAX_EPOCHS,
            head_max_epochs=METHOD1_HEAD_MAX_EPOCHS,
            validation_fraction=METHOD1_VALIDATION_FRACTION,
            early_stopping_patience=METHOD1_EARLY_STOPPING_PATIENCE,
            early_stopping_min_delta=METHOD1_EARLY_STOPPING_MIN_DELTA,
            shared_learning_rate=METHOD1_SHARED_LEARNING_RATE,
            head_learning_rate=METHOD1_HEAD_LEARNING_RATE,
            shared_weight_decay=METHOD1_SHARED_WEIGHT_DECAY,
            head_weight_decay=METHOD1_HEAD_WEIGHT_DECAY,
            batch_size=METHOD1_BATCH_SIZE,
            min_zone_samples=METHOD1_MIN_ZONE_SAMPLES,
            d_model=METHOD1_D_MODEL,
            nhead=METHOD1_NHEAD,
            num_layers=METHOD1_NUM_LAYERS,
            dim_feedforward=METHOD1_DIM_FEEDFORWARD,
            dropout=METHOD1_DROPOUT,
            use_reduce_lr_on_plateau=METHOD1_USE_REDUCE_LR_ON_PLATEAU,
            lr_scheduler_factor=METHOD1_LR_SCHEDULER_FACTOR,
            lr_scheduler_patience=METHOD1_LR_SCHEDULER_PATIENCE,
            lr_scheduler_min_lr=METHOD1_LR_SCHEDULER_MIN_LR,
        )
    else:
        method1_training_artifacts = {'available': False, 'rebuilt': False, 'training_log': pd.DataFrame(), 'stage_paths': method1_stage_paths}
        print('Method 1 checkpoints are missing. Set RUN_METHOD1_TRAINING=True after data prep is available.')

method1_training_log = method1_training_artifacts.get('training_log', pd.DataFrame()).copy()
print(f"Method 1 training stage available: {method1_training_artifacts['available']}; rebuilt: {method1_training_artifacts['rebuilt']}")
if not method1_training_log.empty:
    method1_summary_columns = ['task', 'phase', 'epoch', 'monitored_loss', 'train_loss', 'val_loss', 'learning_rate', 'dropout', 'batch_size', 'lr_scheduler']
    method1_summary_columns = [column for column in method1_summary_columns if column in method1_training_log.columns]
    method1_loss_summary = (
        method1_training_log.sort_values('monitored_loss')
        .groupby(['task', 'phase'], dropna=False, as_index=False)
        .first()[method1_summary_columns]
        .sort_values([column for column in ['task', 'phase'] if column in method1_summary_columns])
    )
else:
    method1_loss_summary = pd.DataFrame()
method1_loss_summary




### Explanation before Method 1 forecast-writing header

This header introduces the stage that turns Method 1 checkpoints into full 2025 forecast parquet files. It clarifies that writing outputs is separate from training.

### 5.D Method 1 Forecast Writing

### Explanation before Method 1 forecast-writing code

This cell optionally writes the full Method 1 2025 forecast parquet files from saved checkpoints. If the full outputs already exist and pass completeness checks, the notebook loads them instead of rewriting tens of millions of rows. This keeps the notebook focused on validation and comparison unless a rebuild is explicitly requested.

In [ ]:
if RUN_METHOD1_FORECAST:
    method1_artifacts = run_direct_bus_forecast_stage(
        force_rebuild=True,  # Rebuild full forecast parquet and metrics from saved checkpoints.
        use_missing_indicators=USE_MISSING_INDICATORS,
    )
elif method1_module.direct_transformer_outputs_complete(method1_paths):
    method1_artifacts = run_direct_bus_forecast_stage(
        force_rebuild=False,  # Reuse existing forecast outputs and metrics.
        use_missing_indicators=USE_MISSING_INDICATORS,
    )
else:
    method1_artifacts = {'method_name': 'method1_direct_bus_transformer', 'paths': method1_paths, 'stage_paths': method1_stage_paths, 'available': False, 'rebuilt': False}
    print('Method 1 forecast outputs are missing. Set RUN_METHOD1_FORECAST=True after checkpoints are available.')

if method1_artifacts.get('available') and 'metrics' in method1_artifacts:
    method1_metrics = method1_artifacts['metrics'].copy()
    if 'method' not in method1_metrics.columns:
        method1_metrics.insert(0, 'method', method1_artifacts['method_name'])
    else:
        method1_metrics['method'] = method1_artifacts['method_name']
else:
    method1_metrics = pd.DataFrame(columns=['method', 'task', 'level', 'model_name', 'n', 'mae', 'rmse', 'wmape', 'sum_actual'])

print(f"Method 1 forecast stage available: {method1_artifacts['available']}; rebuilt: {method1_artifacts['rebuilt']}")
method1_metrics


### Explanation before Method 1 validation header

This header introduces the validation-only step for Method 1. It signals that the following cell checks saved artifacts rather than modifying model logic or retraining.

### 5.E Method 1 Validation

### Explanation before Method 1 validation code

This cell validates Method 1 outputs without retraining. It checks the forecast files, metrics table, training log, missing-bus audit, and sample files, then displays the available Method 1 metrics. This confirms whether Method 1 can be included in the final model comparison.

In [ ]:
method1_validation_artifacts = run_direct_bus_validation_stage()  # Validate existing Method 1 outputs only.
method1_validation = method1_validation_artifacts['validation']  # Keep validation table for review.

if not method1_validation_artifacts['metrics'].empty:
    method1_metrics = method1_validation_artifacts['metrics'].copy()
    if 'method' not in method1_metrics.columns:
        method1_metrics.insert(0, 'method', method1_validation_artifacts['method_name'])
    else:
        method1_metrics['method'] = method1_validation_artifacts['method_name']
    method1_artifacts = method1_validation_artifacts
elif 'method1_metrics' not in globals():
    method1_metrics = pd.DataFrame(columns=['method', 'task', 'level', 'model_name', 'n', 'mae', 'rmse', 'wmape', 'sum_actual'])

method1_validation


### Explanation before Method 2/weather overview

This markdown cell explains the main Method 2 flow: train or load zone forecasts, optionally compare weather-aware forecasts, allocate zone forecasts to bus level using shared bus shares, and validate outputs. It is included to make the multi-stage pipeline easier to follow.

## 6. Method 2 / Weather-Aware Method 2: Zone Forecast + Shared Bus Share

Flow for this section:

1. Train/cache the no-weather Method 2 zone forecasts. The next-day zone model uses 2022-2024 training rows; next-month rolling models use all rows available before each forecast creation cutoff.
2. Train/cache the weather-aware Method 2 zone forecasts with the same cutoff discipline.
3. Compare both zone models at the zone level before bus allocation.
4. Run one shared bus-share pass that writes both no-weather and weather-aware bus-level forecast files.
5. Validate both final output sets.

Cutoff assumptions:

- Task 1 next-day: target day D is forecast at `D-1 00:01`; the zone model and bus-share logic use D-7/D-14/D-28 style historical signals, not D-1 actual load.
- Task 2 next-month: target month is forecast at the first day of the previous month; month-lag features are built only from data available before `forecast_created_at`.

Method 2 uses a two-stage forecasting structure: first forecast load at the zone-hour level, then allocate the zone forecast down to individual buses using historical bus shares. I use this design because the bus-level dataset is very large, sparse, and noisy, while the zone-level load is much more stable and easier to model. Directly predicting every bus-hour can be affected by many zero-load buses, missing buses, new buses, and topology changes. In contrast, zone-level load captures the main system demand pattern more reliably, and bus-share allocation preserves the required bus-level output format.

The zone model is a Transformer-based regression model. Each input feature is treated as a token, such as hour, day of week, month, lagged load, historical average load, and optional weather features. The Transformer can learn interactions among these features instead of using them independently. This is useful because load is not driven by one single variable. For example, the same temperature or same hour can have different effects depending on the month, weekday, holiday status, and recent load pattern.

The no-weather Method 2 is the main reproducible model. It only uses load history and calendar information, so it does not depend on external data downloads. For next-day prediction, it uses historical lag signals such as D-7, D-14, and D-28 instead of D-1 actual load, because the forecast is created at D-1 00:01 and the full D-1 load is not available yet. For next-month prediction, it uses only the data available before the forecast creation cutoff, including previous-year load, month-lag features, and historical month/day/hour averages.

The weather-aware Method 2 is added as an extension because the next-month forecast is harder than the next-day forecast. Month-ahead load depends not only on historical seasonality but also on temperature-driven demand, especially during extreme heat or cold periods. Without weather information, the model can only approximate seasonal demand using past load and calendar features. To improve this, I use the zone names and the bus-to-zone relationship in the provided load data to associate each zone with representative geographic weather points. Then I add hourly weather features such as temperature, dew point, humidity, wind speed, precipitation, cooling degree hours, heating degree hours, and extreme-temperature indicators.

The weather model follows the same cutoff discipline as the no-weather model. For training rows, observed historical weather can be used because it would have been known after the fact. For future target rows, the notebook can use leakage-safe historical weather normals instead of target-year actual weather, unless the weather-oracle setting is explicitly enabled for analysis. This keeps the default workflow realistic while still allowing a separate experiment to show whether weather information could improve the forecast if better weather forecasts were available.

Flow for this section:

Train or reuse the no-weather Method 2 zone forecasts.
Optionally download or reuse the weather cache.
Train or reuse the weather-aware Method 2 zone forecasts.
Compare the no-weather and weather-aware zone forecasts before bus allocation.
Run one shared bus-share allocation pass to produce bus-level forecast files.
Validate both final output sets.

Cutoff assumptions:

Task 1 next-day: target day D is forecast at D-1 00:01. The model and bus-share logic use D-7, D-14, and D-28 style historical signals, not D-1 actual load.
Task 2 next-month: the target month is forecast at the first day of the previous month. Month-lag and historical-average features are built only from data available before forecast_created_at.


### Explanation before no-weather Method 2 header

This header starts the default Method 2 zone-training stage. It distinguishes the primary no-weather pipeline from optional weather experiments.

### 6.A No-Weather Method 2 Zone Training

### Explanation before no-weather Method 2 zone-training code

This cell controls the default no-weather Method 2 zone-level training stage. The goal is to produce one zone-hour forecast for every 2025 target hour before allocating the forecast to buses.

This model is used because zone-level load is more stable than individual bus-level load. A zone aggregates many buses, so missing values, bus additions, bus removals, and small local changes have less impact. This makes the zone model a better place to learn the main demand pattern. After the zone forecast is produced, the bus-share step distributes that forecast back to the required bus-level rows.

The no-weather model uses calendar and historical load features. For next-day forecasting, the most important signals are recent weekly patterns such as D-7, D-14, and D-28, along with historical averages by zone, hour, and calendar category. For next-month forecasting, the model uses previous-year load, month-lag features, and historical month/day/hour averages. These features are built under the forecast cutoff, so the model does not use future target information.

The model is trained separately by zone. This is useful because different zones can have different demand patterns. For example, coastal, urban, and western regions may have different load shapes even under the same calendar conditions. Training by zone allows the model to fit these local patterns while keeping the dataset small enough to train efficiently.

The default path reuses cached zone predictions when they already exist. This avoids retraining expensive Transformer models every time the notebook is run. Since the outputs are deterministic and saved to disk, the notebook can validate the cached forecasts instead of repeating the full training process.

In [ ]:
RUN_METHOD2_ZONE_TRAINING = False  # Formal submission default: reuse cached residual Method 2 zone forecasts.
METHOD2_EPOCHS = 1000  # Maximum epochs for each zone Transformer; early stopping may stop earlier.
METHOD2_VALIDATION_FRACTION = 0.2  # Last 15% of each zone's training timeline is used as validation.
METHOD2_EARLY_STOPPING_PATIENCE = 60  # Stop if monitored validation loss does not improve for this many epochs.
METHOD2_EARLY_STOPPING_MIN_DELTA = 1e-5  # Minimum monitored-loss improvement required to reset early stopping.
METHOD2_LEARNING_RATE = 3e-4  # AdamW initial learning rate for no-weather Method 2.
METHOD2_DROPOUT = 0.1  # Transformer dropout for no-weather Method 2.
METHOD2_USE_REDUCE_LR_ON_PLATEAU = True  # Set True to reduce LR when validation loss plateaus.
METHOD2_LR_SCHEDULER_FACTOR = 0.5  # ReduceLROnPlateau multiplies LR by this factor.
METHOD2_LR_SCHEDULER_PATIENCE = 10  # Plateau epochs before reducing LR.
METHOD2_LR_SCHEDULER_MIN_LR = 1e-5  # Lower bound for scheduler-reduced LR.
METHOD2_BATCH_SIZE = None  # No mini-batch training here: each zone's hourly rows are trained as one full GPU tensor.
METHOD2_NEXT_MONTH_LOOKBACK_MONTHS = [2, 3, 6, 12]  # Next-month zone lag features available before forecast creation: lag_2m, lag_3m, lag_6m, and lag_12m.

import assignment2_method2_zone_share_transformer as method2_module  # Reload Method 2 helpers before running a stage.
method2_module = importlib.reload(method2_module)  # Make sure notebook uses the latest staged functions.
run_method2_zone_forecasting_stage = method2_module.run_method2_zone_forecasting_stage  # Rebind Stage 3.A.
run_method2_bus_share_stage = method2_module.run_method2_bus_share_stage  # Rebind Stage 3.B.
run_method2_validation_stage = method2_module.run_method2_validation_stage  # Rebind Stage 3.C.
run_combined_method2_bus_share_stage = method2_module.run_combined_method2_bus_share_stage  # Rebind combined bus-share runner.

if RUN_METHOD2_ZONE_TRAINING:
    method2_zone_artifacts = run_method2_zone_forecasting_stage(
        force_rebuild=True,  # Retrain and overwrite cached zone predictions.
        epochs=METHOD2_EPOCHS,  # Maximum training epochs.
        validation_fraction=METHOD2_VALIDATION_FRACTION,  # Timeline holdout validation fraction.
        early_stopping_patience=METHOD2_EARLY_STOPPING_PATIENCE,  # Validation-loss early stopping patience.
        early_stopping_min_delta=METHOD2_EARLY_STOPPING_MIN_DELTA,  # Minimum improvement threshold.
        learning_rate=METHOD2_LEARNING_RATE,  # AdamW learning rate.
        dropout=METHOD2_DROPOUT,  # Transformer dropout.
        use_reduce_lr_on_plateau=METHOD2_USE_REDUCE_LR_ON_PLATEAU,  # Optional LR scheduler.
        lr_scheduler_factor=METHOD2_LR_SCHEDULER_FACTOR,  # LR scheduler factor.
        lr_scheduler_patience=METHOD2_LR_SCHEDULER_PATIENCE,  # LR scheduler patience.
        lr_scheduler_min_lr=METHOD2_LR_SCHEDULER_MIN_LR,  # LR scheduler minimum LR.
        next_month_lookback_months=METHOD2_NEXT_MONTH_LOOKBACK_MONTHS,  # Month-lag features for next-month training.
    )
else:
    method2_stage_paths = method2_module.method2_zone_stage_paths(NOTEBOOK_DIR)  # Locate cached zone-stage files.
    if method2_module.method2_zone_outputs_complete(method2_stage_paths):
        method2_zone_artifacts = {
            'method_name': 'method2_zone_forecast_plus_bus_share_transformer',
            'paths': method2_module.output_paths(NOTEBOOK_DIR),
            'stage_paths': method2_stage_paths,
            'next_day_zone': pd.read_parquet(method2_stage_paths['next_day_zone']),
            'next_month_zone': pd.read_parquet(method2_stage_paths['next_month_zone']),
            'training_log': pd.read_csv(method2_stage_paths['training_log']),
            'available': True,
            'rebuilt': False,
            'hyperparameters': {
                'epochs': METHOD2_EPOCHS,
                'validation_fraction': METHOD2_VALIDATION_FRACTION,
                'early_stopping_patience': METHOD2_EARLY_STOPPING_PATIENCE,
                'learning_rate': METHOD2_LEARNING_RATE,
                'dropout': METHOD2_DROPOUT,
                'use_reduce_lr_on_plateau': METHOD2_USE_REDUCE_LR_ON_PLATEAU,
                'batch_size': METHOD2_BATCH_SIZE,
                'next_month_lookback_months': METHOD2_NEXT_MONTH_LOOKBACK_MONTHS,
            },
        }
    else:
        method2_zone_artifacts = {
            'method_name': 'method2_zone_forecast_plus_bus_share_transformer',
            'paths': method2_module.output_paths(NOTEBOOK_DIR),
            'stage_paths': method2_stage_paths,
            'training_log': pd.DataFrame(),
            'available': False,
            'rebuilt': False,
            'hyperparameters': {'batch_size': METHOD2_BATCH_SIZE, 'next_month_lookback_months': METHOD2_NEXT_MONTH_LOOKBACK_MONTHS},
        }
        print('Method 2 zone cache is missing. Set RUN_METHOD2_ZONE_TRAINING=True to train Stage 3.A.')

method2_training_log = method2_zone_artifacts['training_log'].copy()  # Epoch-by-epoch train/validation loss.
print(f"Method 2 zone stage available: {method2_zone_artifacts['available']}; rebuilt: {method2_zone_artifacts['rebuilt']}")
print(f"Method 2 zone training batch size: {METHOD2_BATCH_SIZE} (full-batch per zone)")
print(f"Method 2 next-month lookbacks: {METHOD2_NEXT_MONTH_LOOKBACK_MONTHS}")

if not method2_training_log.empty:  # Summarize best monitored loss for quick tuning review.
    method2_zone_loss_summary = (
        method2_training_log.sort_values('monitored_loss')
        .groupby(['task', 'target_month', 'zone_name'], dropna=False, as_index=False)
        .first()[['task', 'target_month', 'zone_name', 'epoch', 'monitored_loss', 'train_loss', 'val_loss', 'learning_rate', 'dropout', 'lr_scheduler']]
        .sort_values(['task', 'target_month', 'zone_name'])
    )
else:
    method2_zone_loss_summary = pd.DataFrame()
method2_zone_loss_summary.head(30)




### Explanation before optional weather-cache header

This header marks the optional weather-data setup. Keeping it separate prevents the final submission workflow from depending on a network download unless the user deliberately enables it.

### 6.B Optional Weather Data Cache

### Explanation before optional weather-cache code

This cell optionally downloads or reuses the weather-data cache. Weather is treated as an optional extension rather than the default submission path because it adds an external data dependency. Keeping this step behind a flag makes the normal workflow easier to reproduce, while still allowing a weather-aware experiment.

The reason for adding weather is that load forecasting, especially next-month forecasting, is strongly affected by temperature and weather conditions. The no-weather model can learn seasonality from historical load and calendar variables, but it cannot know whether a future month is unusually hot or cold. This limitation is more important for month-ahead prediction because the forecast horizon is longer and recent load lags are less directly informative.

To build weather features, I use the zone information in the provided dataset. Each bus belongs to a zone, and each zone name is mapped to representative geographic points. Hourly weather data is collected for those points and then averaged to the zone-hour level. This creates weather features that can be merged with the zone-level load table by zone_name, target_date, and he.

The cached weather features include temperature, dew point, humidity, wind speed, precipitation, cooling degree hours, heating degree hours, and extreme heat or cold indicators. These features are useful because electricity demand often increases during hot or cold weather due to cooling and heating load. The weather-aware model can therefore learn whether a zone-hour forecast should be adjusted upward or downward relative to the purely historical load pattern.

This cache is optional. If the weather file already exists, the notebook reuses it. If it does not exist and weather downloading is enabled, the notebook downloads the data and writes a local cache. This design prevents repeated network calls and keeps the final pipeline more stable.

In [ ]:
DOWNLOAD_WEATHER = False  # Set False if the weather parquet already exists and you only want to reuse the cache.
if DOWNLOAD_WEATHER:  # Download/refresh zone-level hourly weather from Open-Meteo archive.
    try:  # Try the download first in case meteostat is already installed.
        weather_cache = download_zone_weather(force_download=False, provider='open_meteo')  # Cache file: weather_data/assignment2_zone_hourly_weather_2022_2025.parquet.
    except ImportError:  # If meteostat is missing, install it into the current notebook kernel environment.
        import subprocess  # Used only for optional package installation.
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'meteostat>=1.6'])  # Install the weather downloader dependency.
        weather_module = importlib.reload(weather_module)  # Reload the weather module after installing meteostat.
        download_zone_weather = weather_module.download_zone_weather  # Refresh function reference after reload.
        weather_cache = download_zone_weather(force_download=False, provider='open_meteo')  # Retry the download after installing the dependency.
else:  # Reuse local weather cache paths without making a network request.
    weather_cache = weather_paths(NOTEBOOK_DIR)  # Build expected local weather paths.

print('Weather cache files:')  # Show where the weather files are stored.
for name, path in weather_cache.items():  # Print each weather-related path.
    print(f'  {name}: {path}')  # Display cache path for checking.


### Explanation before weather-aware Method 2 header

This header introduces the optional weather-aware zone model. It makes clear that this is a separate comparison path rather than the default no-weather submission path.

### 6.C Weather-Aware Method 2 Zone Training

### Explanation before weather-aware Method 2 code

This cell controls the optional weather-aware residual zone models. The setting keeps observed 2025 weather disabled by default to avoid leakage; instead, target-year weather normals can be used. This makes the weather experiment more realistic for a forecast setting where future actual weather is not known at forecast creation time.

In [ ]:
RUN_WEATHER_ZONE_TRAINING = False  # Formal submission default: reuse cached weather-aware residual zone forecasts.
USE_OBSERVED_2025_WEATHER = False  # Keep False for leakage-safe target-year weather normals.
WEATHER_EPOCHS = 1000  # Maximum epochs; early stopping may stop earlier.
WEATHER_VALIDATION_FRACTION = 0.2  # Last 15% of each zone's training timeline is used as validation.
WEATHER_EARLY_STOPPING_PATIENCE = 60  # Stop if validation loss does not improve for this many epochs.
WEATHER_EARLY_STOPPING_MIN_DELTA = 1e-5  # Minimum monitored-loss improvement required to reset early stopping.
WEATHER_LEARNING_RATE = 3e-4  # AdamW initial learning rate for the weather-aware zone Transformer.
WEATHER_DROPOUT = 0.10  # Transformer dropout for the weather-aware model.
WEATHER_USE_REDUCE_LR_ON_PLATEAU = True  # Set True to reduce LR when validation loss plateaus.
WEATHER_LR_SCHEDULER_FACTOR = 0.5  # ReduceLROnPlateau multiplies LR by this factor.
WEATHER_LR_SCHEDULER_PATIENCE = 10  # Plateau epochs before reducing LR.
WEATHER_LR_SCHEDULER_MIN_LR = 1e-5  # Lower bound for scheduler-reduced LR.
WEATHER_BATCH_SIZE = None  # No mini-batch training here: each zone's hourly rows are trained as one full GPU tensor.
WEATHER_NEXT_MONTH_LOOKBACK_MONTHS = [2, 3, 6, 12]  # Next-month weather model lag features available before forecast creation: lag_2m, lag_3m, lag_6m, and lag_12m.

import assignment2_weather as weather_module  # Reload the weather module before running this stage.
weather_module = importlib.reload(weather_module)  # Make sure staged helpers are current.
run_weather_zone_training_stage = weather_module.run_weather_zone_training_stage  # Rebind weather Stage 1.
run_weather_bus_share_stage = weather_module.run_weather_bus_share_stage  # Rebind weather Stage 2.
run_weather_validation_stage = weather_module.run_weather_validation_stage  # Rebind weather Stage 3.
weather_model_paths = weather_module.weather_model_paths  # Rebind weather output paths.

if RUN_WEATHER_ZONE_TRAINING:
    weather_zone_artifacts = run_weather_zone_training_stage(
        force_rebuild=True,  # Rebuild cached weather zone predictions after hyperparameter changes.
        force_download=False,  # Reuse cached weather data unless you need to redownload it.
        use_observed_target_weather=USE_OBSERVED_2025_WEATHER,  # False avoids future weather leakage.
        epochs=WEATHER_EPOCHS,  # Maximum training epochs.
        validation_fraction=WEATHER_VALIDATION_FRACTION,  # Timeline holdout validation fraction.
        early_stopping_patience=WEATHER_EARLY_STOPPING_PATIENCE,  # Validation-loss early stopping patience.
        early_stopping_min_delta=WEATHER_EARLY_STOPPING_MIN_DELTA,  # Minimum improvement threshold.
        learning_rate=WEATHER_LEARNING_RATE,  # AdamW learning rate.
        dropout=WEATHER_DROPOUT,  # Transformer dropout.
        use_reduce_lr_on_plateau=WEATHER_USE_REDUCE_LR_ON_PLATEAU,  # Optional LR scheduler.
        lr_scheduler_factor=WEATHER_LR_SCHEDULER_FACTOR,  # LR scheduler factor.
        lr_scheduler_patience=WEATHER_LR_SCHEDULER_PATIENCE,  # LR scheduler patience.
        lr_scheduler_min_lr=WEATHER_LR_SCHEDULER_MIN_LR,  # LR scheduler minimum LR.
        next_month_lookback_months=WEATHER_NEXT_MONTH_LOOKBACK_MONTHS,  # Month-lag features for next-month training.
    )
else:
    weather_paths_dict = weather_model_paths(NOTEBOOK_DIR)  # Locate cached weather model files.
    weather_zone_complete = weather_module.weather_zone_outputs_complete(weather_paths_dict)
    weather_zone_artifacts = {
        'paths': weather_paths_dict,
        'training_log': pd.read_csv(weather_paths_dict['training_log']) if weather_paths_dict['training_log'].exists() else pd.DataFrame(),
        'next_day_zone': pd.read_parquet(weather_paths_dict['next_day_zone']) if weather_zone_complete else pd.DataFrame(),
        'next_month_zone': pd.read_parquet(weather_paths_dict['next_month_zone']) if weather_zone_complete else pd.DataFrame(),
        'available': weather_zone_complete,
        'rebuilt': False,
        'hyperparameters': {
            'epochs': WEATHER_EPOCHS,
            'validation_fraction': WEATHER_VALIDATION_FRACTION,
            'early_stopping_patience': WEATHER_EARLY_STOPPING_PATIENCE,
            'early_stopping_min_delta': WEATHER_EARLY_STOPPING_MIN_DELTA,
            'learning_rate': WEATHER_LEARNING_RATE,
            'dropout': WEATHER_DROPOUT,
            'use_reduce_lr_on_plateau': WEATHER_USE_REDUCE_LR_ON_PLATEAU,
            'batch_size': WEATHER_BATCH_SIZE,
            'next_month_lookback_months': WEATHER_NEXT_MONTH_LOOKBACK_MONTHS,
        },
    }
    if not weather_zone_complete:
        print('Weather zone cache is missing. Set RUN_WEATHER_ZONE_TRAINING=True to train Stage 6.C.')

weather_training_log = weather_zone_artifacts['training_log'].copy()
print(f"Weather zone stage available: {weather_zone_artifacts['available']}; rebuilt: {weather_zone_artifacts['rebuilt']}")
print(f"Weather zone training batch size: {WEATHER_BATCH_SIZE} (full-batch per zone)")
print(f"Weather next-month lookbacks: {WEATHER_NEXT_MONTH_LOOKBACK_MONTHS}")
if not weather_training_log.empty:
    weather_loss_summary = (
        weather_training_log.sort_values('monitored_loss')
        .groupby(['task', 'target_month', 'zone_name'], dropna=False, as_index=False)
        .first()[['task', 'target_month', 'zone_name', 'epoch', 'monitored_loss', 'train_loss', 'val_loss', 'learning_rate', 'dropout', 'lr_scheduler']]
        .sort_values(['task', 'target_month', 'zone_name'])
    )
else:
    weather_loss_summary = pd.DataFrame()
weather_loss_summary.head(30)



### Explanation before zone-diagnostics header

This header introduces pre-allocation zone-level diagnostics. Checking zone accuracy first helps identify whether errors come from the zone model or from bus-share allocation.

### 6.D Zone-Level Forecast Diagnostics Before Bus Share

### Explanation before zone-level diagnostics code

This cell compares the no-weather and weather-aware zone predictions before bus-share allocation. Looking at zone-level metrics first helps separate two sources of error: the zone forecast itself and the later bus allocation step. This diagnostic is useful for deciding which zone model should feed the final bus-level forecast.

In [ ]:
import importlib
import assignment2_method2_zone_share_transformer as method2_module
import assignment2_weather as weather_module

method2_module = importlib.reload(method2_module)
weather_module = importlib.reload(weather_module)
USE_OBSERVED_2025_WEATHER = globals().get("USE_OBSERVED_2025_WEATHER", False)

def _load_method2_zone_artifacts_from_cache():
    stage_paths = method2_module.method2_zone_stage_paths(NOTEBOOK_DIR)
    available = method2_module.method2_zone_outputs_complete(stage_paths)
    return {
        "method_name": method2_module.METHOD_NAME,
        "stage_paths": stage_paths,
        "next_day_zone": pd.read_parquet(stage_paths["next_day_zone"]) if available else pd.DataFrame(),
        "next_month_zone": pd.read_parquet(stage_paths["next_month_zone"]) if available else pd.DataFrame(),
        "training_log": pd.read_csv(stage_paths["training_log"]) if available else pd.DataFrame(),
        "available": available,
        "rebuilt": False,
    }

def _load_weather_zone_artifacts_from_cache():
    paths = weather_module.weather_model_paths(NOTEBOOK_DIR)
    available = weather_module.weather_zone_outputs_complete(paths)
    return {
        "method_name": "method2_zone_share_transformer_with_weather",
        "paths": paths,
        "next_day_zone": pd.read_parquet(paths["next_day_zone"]) if available else pd.DataFrame(),
        "next_month_zone": pd.read_parquet(paths["next_month_zone"]) if available else pd.DataFrame(),
        "training_log": pd.read_csv(paths["training_log"]) if available else pd.DataFrame(),
        "available": available,
        "rebuilt": False,
    }

if "method2_zone_artifacts" not in globals() or not isinstance(method2_zone_artifacts, dict) or not method2_zone_artifacts.get("available"):
    method2_zone_artifacts = _load_method2_zone_artifacts_from_cache()

if "weather_zone_artifacts" not in globals() or not isinstance(weather_zone_artifacts, dict) or not weather_zone_artifacts.get("available"):
    weather_zone_artifacts = _load_weather_zone_artifacts_from_cache()

def _zone_metric_rows(method_label, task, model_name, prediction_frame, actual_frame):
    scored = prediction_frame.copy()
    scored["target_date_str"] = scored["target_date_str"].astype(str)
    scored["he"] = scored["he"].astype("int16")
    scored = scored.merge(actual_frame, on=["target_date_str", "he", "zone_name"], how="left")
    scored["actual_pd"] = scored["actual_pd"].fillna(0.0).astype("float64")

    rows = []
    for zone_name, group in scored.groupby("zone_name", observed=True):
        for label, column in [(model_name, "predict_zone_pd"), ("baseline_zone_pd", "baseline_zone_pd")]:
            error = group[column].astype("float64") - group["actual_pd"]
            rows.append(
                {
                    "method": method_label,
                    "task": task,
                    "zone_name": zone_name,
                    "model_name": label,
                    "n": len(group),
                    "mae": error.abs().mean(),
                    "rmse": (error.pow(2).mean()) ** 0.5,
                    "wmape": error.abs().sum() / group["actual_pd"].abs().sum() if group["actual_pd"].abs().sum() else None,
                    "sum_actual": group["actual_pd"].sum(),
                }
            )
    return rows

zone_actual_2025 = method2_module.load_zone_data(assignment2_data_dir)
zone_actual_2025 = zone_actual_2025[zone_actual_2025["target_date"].dt.year.eq(2025)].copy()
zone_actual_2025["target_date_str"] = zone_actual_2025["target_date"].dt.strftime("%Y-%m-%d")
zone_actual_2025 = zone_actual_2025[["target_date_str", "he", "zone_name", "actual_pd"]]

zone_rows = []
if method2_zone_artifacts.get("available"):
    method2_stage_paths = method2_zone_artifacts.get("stage_paths", method2_module.method2_zone_stage_paths(NOTEBOOK_DIR))
    method2_next_day_name = method2_module.infer_cached_zone_model_name(
        method2_stage_paths,
        "next_day",
        method2_module.TRANSFORMER_NEXT_DAY_MODEL_NAME,
    )
    method2_next_month_name = method2_module.infer_cached_zone_model_name(
        method2_stage_paths,
        "next_month",
        method2_module.TRANSFORMER_NEXT_MONTH_MODEL_NAME,
    )
    zone_rows.extend(
        _zone_metric_rows(
            "method2_zone_share_transformer",
            "next_day",
            method2_next_day_name,
            method2_zone_artifacts["next_day_zone"],
            zone_actual_2025,
        )
    )
    zone_rows.extend(
        _zone_metric_rows(
            "method2_zone_share_transformer",
            "next_month",
            method2_next_month_name,
            method2_zone_artifacts["next_month_zone"],
            zone_actual_2025,
        )
    )
else:
    print("Method 2 zone cache is missing. Run Stage 6.A before this diagnostic cell.")

if weather_zone_artifacts.get("available"):
    weather_paths_dict = weather_zone_artifacts.get("paths", weather_module.weather_model_paths(NOTEBOOK_DIR))
    weather_next_day_default = weather_module.WEATHER_NEXT_DAY_MODEL_NAME
    weather_next_month_default = weather_module.WEATHER_NEXT_MONTH_MODEL_NAME
    weather_next_day_name = method2_module.infer_cached_zone_model_name(weather_paths_dict, "next_day", weather_next_day_default)
    weather_next_month_name = method2_module.infer_cached_zone_model_name(weather_paths_dict, "next_month", weather_next_month_default)
    zone_rows.extend(
        _zone_metric_rows(
            "method2_zone_share_transformer_with_weather",
            "next_day",
            weather_next_day_name,
            weather_zone_artifacts["next_day_zone"],
            zone_actual_2025,
        )
    )
    zone_rows.extend(
        _zone_metric_rows(
            "method2_zone_share_transformer_with_weather",
            "next_month",
            weather_next_month_name,
            weather_zone_artifacts["next_month_zone"],
            zone_actual_2025,
        )
    )
else:
    print("Weather zone cache is missing. Run Stage 6.C before this diagnostic cell.")

zone_metric_columns = ["method", "task", "zone_name", "model_name", "n", "mae", "rmse", "wmape", "sum_actual"]
zone_forecast_by_zone = pd.DataFrame(zone_rows, columns=zone_metric_columns)
if not zone_forecast_by_zone.empty:
    overall_rows = []
    for (method, task, model_name), group in zone_forecast_by_zone.groupby(["method", "task", "model_name"], observed=True):
        n = int(group["n"].sum())
        actual_weight = group["sum_actual"].abs().sum()
        overall_rows.append(
            {
                "method": method,
                "task": task,
                "model_name": model_name,
                "n": n,
                "mae": (group["mae"] * group["n"]).sum() / n if n else None,
                "rmse": ((group["rmse"] ** 2 * group["n"]).sum() / n) ** 0.5 if n else None,
                "wmape": (group["wmape"] * group["sum_actual"].abs()).sum() / actual_weight if actual_weight else None,
                "sum_actual": group["sum_actual"].sum(),
            }
        )
    zone_forecast_overall = pd.DataFrame(overall_rows).sort_values(["task", "wmape", "method", "model_name"]).reset_index(drop=True)
    zone_forecast_overall.to_csv(NOTEBOOK_DIR / "assignment2_zone_forecast_overall_comparison.csv", index=False)
    zone_forecast_by_zone.to_csv(NOTEBOOK_DIR / "assignment2_zone_forecast_by_zone_comparison.csv", index=False)
else:
    zone_forecast_overall = pd.DataFrame(columns=["method", "task", "model_name", "n", "mae", "rmse", "wmape", "sum_actual"])

print("Overall zone-level forecast comparison")
display(zone_forecast_overall)
print("Per-zone zone-level forecast comparison")
zone_forecast_by_zone.sort_values(["task", "zone_name", "wmape", "method", "model_name"], na_position="last")


### Explanation before shared bus-share allocation header

This header introduces the stage that converts zone-level predictions into the required bus-level forecast files. It highlights that the same allocation method is used for fair comparison across zone-model variants.

### 6.E Shared Bus-Share Allocation for No-Weather and Weather-Aware Method 2

### Explanation before shared bus-share allocation code

This cell runs the shared bus-share allocation stage for both the no-weather and weather-aware Method 2 forecasts. At this point, the model has already produced zone-hour forecasts. This cell converts those zone-level forecasts into the required bus-level forecast files.

The reason for using a shared allocation step is fairness. The no-weather and weather-aware models should be compared mainly by the quality of their zone forecasts, not by different bus distribution rules. Therefore, both model variants use the same bus-share logic. The only major difference between them is whether the zone forecast includes weather features.

The bus-share allocation works by distributing each predicted zone-hour load to buses inside the same zone and hour. For each target date, he, and zone_name, the pipeline first reads the 2025 bus rows that must receive forecasts. Then it calculates each bus historical share within that zone-hour.

Bus share is calculated as:

bus_share = historical_bus_load / historical_total_zone_load

More specifically:

bus_share for bus b in zone z at hour h
= historical load of bus b in zone z at hour h
  / historical total load of all buses in zone z at hour h

After the shares are calculated, they are normalized within each date / he / zone_name group, so the available bus shares in the same zone-hour add up to 1 when reliable historical information exists.

The final bus-level forecast is calculated as:

bus_forecast = zone_forecast * bus_share

More specifically:

forecast load for bus b in zone z at hour h
= predicted total load for zone z at hour h
  * historical share of bus b in that zone-hour

For the next-day forecast, the share is based on recent weekly history. The pipeline uses weighted historical share sources from D-7, D-14, and D-28. This is consistent with the forecast cutoff because the target day D is forecast at D-1 00:01, so the full D-1 actual load should not be used. Weekly lags are used instead because bus-level load distribution often follows weekly patterns.

For the next-month forecast, the share is based on a same-season previous-year window. The pipeline starts from the same calendar date in the previous year and also uses nearby dates such as -14, -7, +7, and +14 days. This makes the allocation less sensitive to one noisy historical day and better aligned with seasonal bus-level patterns.

If a task-specific share is missing, the pipeline falls back to a 2024 bus-hour average share. This fallback estimates how much each bus usually contributes to its zone at the same hour. If a zone-hour group still has no reliable historical share, the bus share remains zero instead of forcing an equal split. This avoids assigning artificial load to buses that are all-missing, inactive, or have no historical nonzero load evidence.

The cell also handles missing and sparse bus history. Partial-missing buses with some nonzero historical load can use the historical imputation logic built earlier. All-missing buses are excluded from share estimation because their historical pd does not provide reliable evidence of load. These cases are also counted in the missing-bus audit output.

Finally, this cell writes the required bus-level forecast parquet files, writes sample CSV files, and computes bus-level and zone-level metrics. The zone-level metrics after allocation are useful because the sum of bus forecasts within each zone-hour should approximately reproduce the original zone forecast, except in zero-share cases where no reliable bus distribution information exists.


In [ ]:
import importlib
import assignment2_method2_zone_share_transformer as method2_module
import assignment2_weather as weather_module

method2_module = importlib.reload(method2_module)
weather_module = importlib.reload(weather_module)
run_method2_zone_forecasting_stage = method2_module.run_method2_zone_forecasting_stage
run_method2_bus_share_stage = method2_module.run_method2_bus_share_stage
run_method2_validation_stage = method2_module.run_method2_validation_stage
run_combined_method2_bus_share_stage = method2_module.run_combined_method2_bus_share_stage
run_weather_zone_training_stage = weather_module.run_weather_zone_training_stage
run_weather_bus_share_stage = weather_module.run_weather_bus_share_stage
run_weather_validation_stage = weather_module.run_weather_validation_stage
weather_model_paths = weather_module.weather_model_paths

# Fast guard: catch stale report/model-name issues before the expensive bus-share pass.
_method2_report_preflight_paths = method2_module.output_paths(NOTEBOOK_DIR).copy()
if _method2_report_preflight_paths["metrics"].exists():
    _tmp_report_path = NOTEBOOK_DIR / "_tmp_method2_report_preflight.md"
    _method2_report_preflight_paths["report"] = _tmp_report_path
    _existing_method2_metrics = pd.read_csv(_method2_report_preflight_paths["metrics"])
    method2_module.write_report(_method2_report_preflight_paths, assignment2_data_dir, _existing_method2_metrics, 0.0)
    _tmp_report_path.unlink(missing_ok=True)
    print("Method 2 report preflight passed with", method2_module.__file__)

RUN_COMBINED_METHOD2_BUS_SHARE = False  # Formal submission default: do not rerun the 88M-row bus-share pass.
RUN_WEATHER_BUS_SHARE = False  # Legacy weather-only pass; keep False to avoid repeating the 88M-row bus-share stage.
COMBINED_FORECAST_WORKERS = 8  # 64GB RAM setting; reduce to 4-6 if the machine starts swapping.

if RUN_COMBINED_METHOD2_BUS_SHARE:
    combined_bus_artifacts = run_combined_method2_bus_share_stage(
        force_rebuild=False,  # Set True to retrain both no-weather and weather-aware bus-share forecasts together after zone-level changes.
        forecast_workers=COMBINED_FORECAST_WORKERS,
        include_weather=True,
        use_observed_target_weather=USE_OBSERVED_2025_WEATHER,
    )
    method2_validation = combined_bus_artifacts.get('validation')
    if method2_validation is None:
        method2_validation = run_method2_validation_stage()['validation']
    weather_validation = combined_bus_artifacts.get('weather_validation')
    if weather_validation is None and 'weather_paths' in combined_bus_artifacts:
        weather_validation = run_weather_validation_stage()['validation']

    method2_artifacts = {
        'method_name': method2_module.METHOD_NAME,
        'paths': combined_bus_artifacts['paths'],
        'metrics': combined_bus_artifacts['metrics'],
        'validation': method2_validation,
        'available': True,
        'rebuilt': combined_bus_artifacts.get('rebuilt', True),
    }
    method2_metrics = combined_bus_artifacts['metrics'].copy()
    if 'method' not in method2_metrics.columns:
        method2_metrics.insert(0, 'method', method2_module.METHOD_NAME)
    else:
        method2_metrics['method'] = method2_module.METHOD_NAME

    weather_artifacts = {
        'method_name': 'method2_zone_share_transformer_with_weather',
        'paths': combined_bus_artifacts['weather_paths'],
        'metrics': combined_bus_artifacts['weather_metrics'],
        'validation': weather_validation,
        'available': True,
        'rebuilt': combined_bus_artifacts.get('rebuilt', True),
    }
    weather_metrics = combined_bus_artifacts['weather_metrics'].copy()
    if 'method' not in weather_metrics.columns:
        weather_metrics.insert(0, 'method', weather_artifacts['method_name'])
    else:
        weather_metrics['method'] = weather_artifacts['method_name']
elif RUN_WEATHER_BUS_SHARE:
    weather_artifacts = run_weather_bus_share_stage(
        force_rebuild=True,
        use_observed_target_weather=USE_OBSERVED_2025_WEATHER,
        forecast_workers=COMBINED_FORECAST_WORKERS,
    )
    weather_metrics = weather_artifacts['metrics'].copy()
    if 'method' not in weather_metrics.columns:
        weather_metrics.insert(0, 'method', weather_artifacts['method_name'])
    else:
        weather_metrics['method'] = weather_artifacts['method_name']
else:
    existing_method2_metrics = NOTEBOOK_DIR / 'assignment2_metrics.csv'
    method2_paths = method2_module.output_paths(NOTEBOOK_DIR)
    if existing_method2_metrics.exists():
        method2_metrics = pd.read_csv(existing_method2_metrics)
        if 'method' not in method2_metrics.columns:
            method2_metrics.insert(0, 'method', method2_module.METHOD_NAME)
        else:
            method2_metrics['method'] = method2_module.METHOD_NAME
        method2_artifacts = {'method_name': method2_module.METHOD_NAME, 'paths': method2_paths, 'metrics': method2_metrics, 'available': True, 'rebuilt': False}
    else:
        method2_metrics = pd.DataFrame()
        method2_artifacts = {'method_name': method2_module.METHOD_NAME, 'paths': method2_paths, 'metrics': method2_metrics, 'available': False, 'rebuilt': False}
        print('Method 2 bus-share allocation skipped and existing metrics were not found.')

    existing_weather_metrics = NOTEBOOK_DIR / 'assignment2_weather_metrics.csv'
    if existing_weather_metrics.exists():
        weather_metrics = pd.read_csv(existing_weather_metrics)
        if 'method' not in weather_metrics.columns:
            weather_metrics.insert(0, 'method', 'method2_zone_share_transformer_with_weather')
        else:
            weather_metrics['method'] = 'method2_zone_share_transformer_with_weather'
        weather_artifacts = {'method_name': 'method2_zone_share_transformer_with_weather', 'paths': weather_model_paths(NOTEBOOK_DIR), 'metrics': weather_metrics, 'available': True, 'rebuilt': False}
    else:
        weather_metrics = pd.DataFrame()
        weather_artifacts = {'method_name': 'method2_zone_share_transformer_with_weather', 'paths': weather_model_paths(NOTEBOOK_DIR), 'metrics': weather_metrics, 'available': False, 'rebuilt': False}
        print('Weather-aware bus-share allocation skipped and existing metrics were not found.')
weather_metrics


### Explanation before Method 2 validation header

This header introduces validation for Method 2 and optional weather-aware deliverables. It separates output checking from training and forecast generation.

### 6.F Method 2 and Weather-Aware Validation

### Explanation before Method 2 validation code

This cell validates the Method 2 deliverables after bus-share allocation. It checks whether the expected forecast files and supporting artifacts are complete, then displays the validation checklist and metrics. This step is necessary before using Method 2 as the final submission model.

In [ ]:
if method2_module.outputs_are_complete(method2_artifacts["paths"], assignment2_data_dir):
    method2_validation_artifacts = run_method2_validation_stage()
    method2_validation = method2_validation_artifacts["validation"]
    if not method2_validation_artifacts["metrics"].empty:
        method2_metrics = method2_validation_artifacts["metrics"].copy()
        if "method" not in method2_metrics.columns:
            method2_metrics.insert(0, "method", method2_validation_artifacts["method_name"])
        else:
            method2_metrics["method"] = method2_validation_artifacts["method_name"]
        method2_artifacts = method2_validation_artifacts
else:
    method2_validation_artifacts = {"method_name": method2_module.METHOD_NAME, "metrics": pd.DataFrame(), "validation": pd.DataFrame()}
    method2_validation = pd.DataFrame(
        [{"check": "method2_outputs_missing", "passed": False, "detail": "Run the shared bus allocation cell before validation."}]
    )

if "weather_artifacts" in globals() and weather_artifacts.get("available"):
    weather_validation_artifacts = run_weather_validation_stage()
    weather_validation = weather_validation_artifacts["validation"]
    if not weather_validation_artifacts["metrics"].empty:
        weather_metrics = weather_validation_artifacts["metrics"].copy()
        if "method" not in weather_metrics.columns:
            weather_metrics.insert(0, "method", weather_validation_artifacts["method_name"])
        else:
            weather_metrics["method"] = weather_validation_artifacts["method_name"]
        weather_artifacts = weather_validation_artifacts
else:
    weather_validation_artifacts = {"method_name": "method2_zone_share_transformer_with_weather", "metrics": pd.DataFrame(), "validation": pd.DataFrame()}
    weather_validation = pd.DataFrame(
        [{"check": "weather_outputs_missing", "passed": False, "detail": "Run the shared bus allocation cell with include_weather=True before validation."}]
    )

validation_summary = pd.concat(
    [
        method2_validation.assign(method="method2_zone_share_transformer"),
        weather_validation.assign(method="method2_zone_share_transformer_with_weather"),
    ],
    ignore_index=True,
    sort=False,
)
validation_summary

### Explanation before optional/experimental section

This markdown cell states that optional experiments are not part of the default final submission unless they pass validation. This prevents exploratory results from being accidentally treated as official deliverables.

### Explanation before combined-metrics section header

This header introduces the final comparison table. At this point, the notebook has already collected metrics from each available method, so the next step is to compare them side by side.

## 8. Combined Metrics and Final Model Selection

### Explanation before combined metrics and model-selection code

This cell combines metrics from Method 1, no-weather Method 2, and weather-aware Method 2 when available. It then pivots the results for easier comparison across task, level, and model. This makes final model selection evidence-based rather than relying on a single isolated metric.

In [ ]:
metric_frames = []
for frame in [method1_metrics, method2_metrics, weather_metrics if "weather_metrics" in globals() else pd.DataFrame()]:
    if frame is not None and not frame.empty:
        metric_frames.append(frame.copy())

combined_metrics = pd.concat(metric_frames, ignore_index=True, sort=False) if metric_frames else pd.DataFrame()
summary_columns = [col for col in ["method", "task", "level", "model_name", "n", "mae", "rmse", "wmape", "sum_actual"] if col in combined_metrics.columns]
combined_metrics_sorted = combined_metrics[summary_columns].sort_values(["task", "level", "model_name"], na_position="last") if not combined_metrics.empty else combined_metrics

if not combined_metrics.empty:
    forecast_model_rows = combined_metrics[~combined_metrics["model_name"].astype(str).str.startswith("baseline_")].copy()
    best_per_task = (
        forecast_model_rows
        .sort_values(["task", "level", "wmape"], na_position="last")
        .groupby(["task", "level"], as_index=False)
        .first()[summary_columns]
    )
else:
    best_per_task = pd.DataFrame(columns=summary_columns)

print("Combined metrics sorted by task, level, model_name")
display(combined_metrics_sorted)
print("Best forecast model per task and level by WMAPE")
best_per_task

### Explanation before final-deliverables section header

This header introduces the final artifact checklist. It focuses the reviewer on the files that need to be uploaded to GitHub or included in the submission package.

## 9. Final Deliverables

### Explanation before final deliverables code

This cell lists the main files that should be included in the final submission and performs lightweight readability/schema checks on parquet deliverables. It provides a final checklist for GitHub upload so missing files, broken paths, or incorrect schemas can be found before submission.

In [ ]:
def _parquet_detail(path):
    if not path.exists() or path.suffix.lower() != ".parquet":
        return {"rows": None, "schema_ok": None, "readable_sample": None, "error": None}
    try:
        pf = pq.ParquetFile(path)
        schema_ok = pf.schema_arrow.names == forecast_schema
        readable_sample = True
        try:
            sample_columns = [col for col in ["model_name", "predict_pd"] if col in pf.schema_arrow.names]
            if pf.num_row_groups and sample_columns:
                pf.read_row_group(0, columns=sample_columns)
        except Exception as sample_exc:
            readable_sample = False
            return {"rows": pf.metadata.num_rows, "schema_ok": schema_ok, "readable_sample": readable_sample, "error": repr(sample_exc)}
        return {"rows": pf.metadata.num_rows, "schema_ok": schema_ok, "readable_sample": readable_sample, "error": None}
    except Exception as exc:
        return {"rows": None, "schema_ok": False, "readable_sample": False, "error": repr(exc)}

required_keys = ["next_day", "next_month", "metrics", "report", "ai_log"]
optional_keys = ["next_day_sample", "next_month_sample", "zone_accuracy_by_zone", "missingness", "missing_bus_audit"]

def _collect_deliverables(method_label, artifacts, required=True):
    rows = []
    paths = artifacts.get("paths", {}) if artifacts else {}
    for key in required_keys + optional_keys:
        if key not in paths:
            continue
        path = Path(paths[key])
        parquet_detail = _parquet_detail(path)
        rows.append(
            {
                "method": method_label,
                "file_key": key,
                "required": key in required_keys and required,
                "exists": path.exists(),
                "rows": parquet_detail.get("rows"),
                "schema_ok": parquet_detail.get("schema_ok"),
                "readable_sample": parquet_detail.get("readable_sample"),
                "error": parquet_detail.get("error"),
                "path": str(path),
            }
        )
    return rows

deliverable_rows = []
deliverable_rows.extend(_collect_deliverables("method1_direct_bus_transformer", method1_artifacts, required=True))
deliverable_rows.extend(_collect_deliverables("method2_zone_share_transformer", method2_artifacts, required=True))
if "weather_artifacts" in globals():
    weather_required = bool(weather_validation["passed"].all()) if "weather_validation" in globals() and not weather_validation.empty else False
    deliverable_rows.extend(_collect_deliverables("method2_zone_share_transformer_with_weather", weather_artifacts, required=weather_required))

deliverables = pd.DataFrame(deliverable_rows)
validation_status = pd.DataFrame(
    [
        {"method": "method1_direct_bus_transformer", "all_validation_passed": bool(method1_validation["passed"].all()) if "method1_validation" in globals() and not method1_validation.empty else False},
        {"method": "method2_zone_share_transformer", "all_validation_passed": bool(method2_validation["passed"].all()) if "method2_validation" in globals() and not method2_validation.empty else False},
        {"method": "method2_zone_share_transformer_with_weather", "all_validation_passed": bool(weather_validation["passed"].all()) if "weather_validation" in globals() and not weather_validation.empty else False},
    ]
)

print("Validation status")
display(validation_status)
print("Final deliverable files")
deliverables.sort_values(["method", "required", "file_key"], ascending=[True, False, True])